# Day 23 — Prediction API Integration (ML + DL)

**Project:** IPL Player Performance Analytics and Prediction  
**Goal:** Audit the trained model artifacts and feature contracts, then create a safe prediction API integration plan inside the existing Django app.

## Prediction targets
- `next_match_runs`
- `next_match_wickets`
- `next_match_runs_conceded`

## API routes planned
- `POST /api/predict/batting/`
- `POST /api/predict/bowling/`
- `POST /api/predict/`
- Batch prediction remains a later integration step.

**Important:** This notebook does not guess model filenames, feature columns, input shapes, or artifact formats. It audits files under `models/`, `reports/`, and `data/processed/`, reads available model metadata, and creates a prediction integration inventory. It generates a separate implementation module and URL patch only after inspecting available artifacts. Existing `views.py` and `urls.py` are preserved.

A trained model is not ready for API use until its exact ordered feature list, preprocessing requirements, target, and input schema are known.

In [1]:
from pathlib import Path
import os, sys, subprocess, json, csv, re, shutil
from datetime import datetime

REPO_ROOT = Path.cwd().resolve()
if not (REPO_ROOT / "django_api").is_dir():
    candidates = [p for p in [REPO_ROOT, *REPO_ROOT.parents] if (p / "django_api").is_dir()]
    if candidates:
        REPO_ROOT = candidates[0]
DJANGO_ROOT = REPO_ROOT / "django_api"
APP_DIR = DJANGO_ROOT / "ipl_api"
MANAGE_PY = DJANGO_ROOT / "manage.py"
VIEWS_PATH = APP_DIR / "views.py"
URLS_PATH = APP_DIR / "urls.py"
MODEL_DIR = REPO_ROOT / "models"
REPORTS_DIR = REPO_ROOT / "reports"
PROCESSED_DIR = REPO_ROOT / "data" / "processed"
DAY23_REPORT_DIR = REPORTS_DIR / "day23"
DOCS_DIR = REPO_ROOT / "docs"
DAY23_REPORT_DIR.mkdir(parents=True, exist_ok=True)
DOCS_DIR.mkdir(parents=True, exist_ok=True)

assert MANAGE_PY.exists(), f"Could not find {MANAGE_PY}; run from the repository root."
assert VIEWS_PATH.exists(), f"Could not find {VIEWS_PATH}."
assert URLS_PATH.exists(), f"Could not find {URLS_PATH}."
print("Repository:", REPO_ROOT)
print("Django root:", DJANGO_ROOT)
print("Model directory exists:", MODEL_DIR.exists())
print("Existing views.py and urls.py will be preserved.")


Repository: D:\Git\Personal\ipl-player-performance
Django root: D:\Git\Personal\ipl-player-performance\django_api
Model directory exists: True
Existing views.py and urls.py will be preserved.


In [2]:
# Inventory model artifacts, reports, feature datasets and any saved metadata.
artifact_extensions = {".joblib", ".pkl", ".pickle", ".keras", ".h5", ".json", ".yaml", ".yml", ".txt", ".csv", ".parquet"}
roots = [MODEL_DIR, REPORTS_DIR, PROCESSED_DIR]
inventory = []
for root in roots:
    if root.exists():
        for path in root.rglob("*"):
            if path.is_file() and (path.suffix.lower() in artifact_extensions or path.name.lower() in {"feature_names.txt", "metadata.json"}):
                try:
                    size = path.stat().st_size
                except OSError:
                    size = None
                inventory.append({
                    "root": str(root.relative_to(REPO_ROOT)),
                    "path": str(path.relative_to(REPO_ROOT)),
                    "suffix": path.suffix.lower(),
                    "size_bytes": size
                })
inventory.sort(key=lambda x: x["path"].lower())
with (DAY23_REPORT_DIR / "artifact_inventory.csv").open("w", newline="", encoding="utf-8") as f:
    w = csv.DictWriter(f, fieldnames=["root", "path", "suffix", "size_bytes"])
    w.writeheader()
    w.writerows(inventory)
print("Artifact files found:", len(inventory))
for item in inventory[:150]:
    print(f'{item["path"]} | {item["size_bytes"]} bytes')
if len(inventory) > 150:
    print(f"... {len(inventory)-150} additional files recorded in artifact_inventory.csv")


Artifact files found: 114
data\processed\batting_next_match_features.csv | 8856766 bytes
data\processed\batting_player_match.csv | 4529086 bytes
data\processed\bowling_next_match_features.csv | 6741508 bytes
data\processed\bowling_player_match.csv | 3410075 bytes
data\processed\day11_sequences\batting_next_match_runs_features.json | 913 bytes
data\processed\day11_sequences\batting_next_match_runs_metadata.csv | 1227867 bytes
data\processed\day11_sequences\bowling_next_match_runs_conceded_features.json | 940 bytes
data\processed\day11_sequences\bowling_next_match_runs_conceded_metadata.csv | 1070958 bytes
data\processed\day11_sequences\bowling_next_match_wickets_features.json | 934 bytes
data\processed\day11_sequences\bowling_next_match_wickets_metadata.csv | 996948 bytes
data\processed\match_metadata.csv | 143556 bytes
data\processed\match_metadata_2026.csv | 15738 bytes
data\processed\player_registry_observations.csv | 1583770 bytes
models\day10\batting_next_match_runs_candidate.jobli

In [3]:
# Inspect prediction feature datasets without assuming exact columns beyond known targets.
feature_files = [
    PROCESSED_DIR / "batting_next_match_features.csv",
    PROCESSED_DIR / "bowling_next_match_features.csv",
]
dataset_summary = []
for path in feature_files:
    if not path.exists():
        dataset_summary.append({"path": str(path.relative_to(REPO_ROOT)), "exists": False, "rows": None, "columns": None, "targets_present": False})
        print("MISSING:", path)
        continue
    import pandas as pd
    df = pd.read_csv(path, nrows=5)
    # Get full row count efficiently by reading only the CSV header and counting lines.
    with path.open("r", encoding="utf-8-sig", errors="replace") as handle:
        row_count = max(sum(1 for _ in handle) - 1, 0)
    targets = ["next_match_runs", "next_match_wickets", "next_match_runs_conceded"]
    summary = {
        "path": str(path.relative_to(REPO_ROOT)),
        "exists": True,
        "rows": row_count,
        "columns": list(df.columns),
        "targets_present": [t for t in targets if t in df.columns],
        "sample_rows_read": len(df),
    }
    dataset_summary.append(summary)
    print(json.dumps(summary, indent=2, default=str))
(DAY23_REPORT_DIR / "feature_dataset_summary.json").write_text(json.dumps(dataset_summary, indent=2, default=str), encoding="utf-8")


{
  "path": "data\\processed\\batting_next_match_features.csv",
  "exists": true,
  "rows": 18030,
  "columns": [
    "match_id",
    "season",
    "venue",
    "city",
    "team_1",
    "team_2",
    "match_type",
    "team",
    "opposition",
    "player_name",
    "player_id",
    "player_name_norm",
    "team_norm",
    "opposition_norm",
    "runs",
    "balls_faced",
    "fours",
    "sixes",
    "batting_position",
    "was_dismissed",
    "dismissal_kind",
    "innings_count",
    "strike_rate",
    "match_date",
    "season_meta",
    "venue_meta",
    "city_meta",
    "team_1_meta",
    "team_2_meta",
    "date",
    "next_match_runs",
    "next_match_id",
    "next_match_date",
    "matches_seen_before",
    "prior_career_batting_runs",
    "prior_last5_mean_batting_runs",
    "prior_last5_count_batting_runs",
    "prior_career_batting_balls_faced",
    "prior_last5_mean_batting_balls_faced",
    "prior_last5_count_batting_balls_faced",
    "prior_career_batting_fours",
    

3102

In [4]:
# Inspect Day 10–14 reports for model selection, metrics and artifact readiness.
report_matches = []
if REPORTS_DIR.exists():
    for path in REPORTS_DIR.rglob("*"):
        if path.is_file() and any(term in path.name.lower() for term in ["model", "metric", "comparison", "candidate", "readiness", "feature", "sequence", "deep_learning"]):
            report_matches.append(str(path.relative_to(REPO_ROOT)))
with (DAY23_REPORT_DIR / "model_related_reports.txt").open("w", encoding="utf-8") as f:
    f.write("\n".join(sorted(report_matches)))
print("Model-related report files:")
for item in sorted(report_matches):
    print(item)


Model-related report files:
reports\day10\final_candidate_test_metrics.csv
reports\day10\final_candidate_test_predictions.csv
reports\day10\model_manifest.csv
reports\day11\sequence_manifest.csv
reports\day11\sequence_validation_checks.csv
reports\day12\model_artifacts_manifest.csv
reports\day12\selected_lstm_gru_test_metrics.csv
reports\day12\validation_architecture_comparison.csv
reports\day13\comparison_interpretation.csv
reports\day13\ml_vs_dl_model_comparison.csv
reports\day14\candidate_readiness.csv
reports\day14\deep_learning_error_summary.csv
reports\day14\largest_deep_learning_errors.csv
reports\day14\metric_consistency_check.csv
reports\day15\candidate_key_checks.csv
reports\day16\candidate_key_checks.csv
reports\day23\feature_dataset_summary.json
reports\day5\batting_feature_correlations.csv
reports\day5\bowling_feature_correlations.csv
reports\day7\baseline_model_metrics.csv
reports\day7\data_readiness_audit.csv
reports\day7\feature_eligibility_review.csv
reports\day8\featu

In [5]:
# Generate a model readiness checklist based on actual discovered files.
model_files = [x for x in inventory if x["root"].startswith("models")]
feature_metadata_files = [x for x in inventory if any(term in x["path"].lower() for term in ["feature", "metadata", "manifest"])]
readiness = {
    "model_files_found": len(model_files),
    "feature_or_metadata_files_found": len(feature_metadata_files),
    "batting_feature_dataset_exists": (PROCESSED_DIR / "batting_next_match_features.csv").exists(),
    "bowling_feature_dataset_exists": (PROCESSED_DIR / "bowling_next_match_features.csv").exists(),
    "required_targets": ["next_match_runs", "next_match_wickets", "next_match_runs_conceded"],
    "api_readiness_checks": [
        {"check": "exact artifact path and format known", "status": "review inventory"},
        {"check": "feature names and order saved with model", "status": "must verify"},
        {"check": "preprocessing pipeline serialized with model", "status": "must verify"},
        {"check": "target and model type identified", "status": "must verify"},
        {"check": "training/test split and metrics documented", "status": "review reports"},
        {"check": "API input validation defined", "status": "pending"},
        {"check": "single and batch predictions tested", "status": "pending"},
    ]
}
(DAY23_REPORT_DIR / "prediction_readiness.json").write_text(json.dumps(readiness, indent=2), encoding="utf-8")
print(json.dumps(readiness, indent=2))


{
  "model_files_found": 22,
  "feature_or_metadata_files_found": 23,
  "batting_feature_dataset_exists": true,
  "bowling_feature_dataset_exists": true,
  "required_targets": [
    "next_match_runs",
    "next_match_wickets",
    "next_match_runs_conceded"
  ],
  "api_readiness_checks": [
    {
      "check": "exact artifact path and format known",
      "status": "review inventory"
    },
    {
      "check": "feature names and order saved with model",
      "status": "must verify"
    },
    {
      "check": "preprocessing pipeline serialized with model",
      "status": "must verify"
    },
    {
      "check": "target and model type identified",
      "status": "must verify"
    },
    {
      "check": "training/test split and metrics documented",
      "status": "review reports"
    },
    {
      "check": "API input validation defined",
      "status": "pending"
    },
    {
      "check": "single and batch predictions tested",
      "status": "pending"
    }
  ]
}


In [6]:
# Run Django check and Day 21 regression tests before adding prediction routes.
def run_cmd(args):
    p = subprocess.run(args, cwd=str(DJANGO_ROOT), text=True, capture_output=True)
    print("$", " ".join(map(str, args)))
    print("Return code:", p.returncode)
    print(p.stdout)
    if p.stderr:
        print(p.stderr)
    return p

check = run_cmd([sys.executable, str(MANAGE_PY), "check"])
regression = run_cmd([sys.executable, str(MANAGE_PY), "test", "ipl_api.tests.test_pagination_utils_day21", "--verbosity", "1"])


$ c:\Users\Vinod\anaconda3\envs\ipl-player-performance\python.exe D:\Git\Personal\ipl-player-performance\django_api\manage.py check
Return code: 0
System check identified no issues (0 silenced).

$ c:\Users\Vinod\anaconda3\envs\ipl-player-performance\python.exe D:\Git\Personal\ipl-player-performance\django_api\manage.py test ipl_api.tests.test_pagination_utils_day21 --verbosity 1
Return code: 0
Found 6 test(s).
System check identified no issues (0 silenced).

......
----------------------------------------------------------------------
Ran 6 tests in 0.000s

OK



In [7]:
# Write Day 23 audit report; no prediction code is generated unless artifact contracts are established.
doc = [
    "# Day 23 — Prediction API Integration Audit",
    "",
    "- Generated: " + datetime.now().isoformat(timespec="seconds"),
    "- Model artifacts inventoried: " + str(len(model_files)),
    "- Django check return code: " + str(check.returncode),
    "- Day 21 regression return code: " + str(regression.returncode),
    "",
    "## Prediction targets",
    "",
    "- `next_match_runs`",
    "- `next_match_wickets`",
    "- `next_match_runs_conceded`",
    "",
    "## Files to review",
    "",
    "- `reports/day23/artifact_inventory.csv`",
    "- `reports/day23/feature_dataset_summary.json`",
    "- `reports/day23/model_related_reports.txt`",
    "- `reports/day23/prediction_readiness.json`",
    "",
    "## Readiness rule",
    "",
    "Do not load a model in an API view until the exact model artifact, target, ordered feature list, preprocessing pipeline, and required input schema are confirmed. If these are missing, keep the endpoint unavailable with an explicit response instead of returning fabricated predictions.",
    "",
    "## Status",
    "",
    "This notebook audits the existing project and validates Django/regression tests. It does not yet implement prediction endpoints because their implementation must be based on the real artifacts and feature contracts found by this audit.",
]
(DOCS_DIR / "day23_prediction_api_audit.md").write_text("\n".join(doc) + "\n", encoding="utf-8")
print("Saved:", DOCS_DIR / "day23_prediction_api_audit.md")


Saved: D:\Git\Personal\ipl-player-performance\docs\day23_prediction_api_audit.md


## Day 23 completion checklist

- [ ] Run artifact inventory.
- [ ] Confirm each selected model artifact's path and format.
- [ ] Confirm exact ordered feature list and preprocessing.
- [ ] Confirm target/model mapping for batting and bowling.
- [ ] Confirm Django check and Day 21 regression tests pass.
- [ ] Based on the real artifact contract, implement prediction serializers/views and register routes.
- [ ] Test valid input, missing features, invalid types, and artifact-not-found behavior.
- [ ] Compare API prediction to direct model prediction for the same input.

**Do not mark Day 23 complete until prediction endpoints are implemented and tested.**